In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect("practice.db")
con.executescript(open("practice_setup.sql").read())

In [2]:
def run(q):
    return pd.read_sql(q, con)

In [3]:
"""-- 1. Basic filter + sort  [Service / screening]
-- Names and salaries of employees earning more than 65000,
-- highest salary first."""
query1 = 'Select name,salary from employees where salary > 65000 order by salary desc'
run(query1)

,name,salary
0,Arjun,95000
1,Sneha,88000
2,Divya,81000
3,Kavya,76000
4,Priya,72000
5,Ramesh,70000
6,Karan,68000


In [4]:
"""-- 2. INNER JOIN  [Service / screening]
-- Each employee's name alongside their dept_name.
-- (Think: who disappears here and why?)"""
query2 = 'Select e.name,d.dept_name from employees e INNER JOIN departments d on d.dept_id=e.dept_id'
run(query2)
# Nisha will disappear as her dept_id in not in the departments table

,name,dept_name
0,Arjun,Engineering
1,Priya,Engineering
2,Karan,Engineering
3,Divya,Engineering
4,Rohit,Sales
5,Sneha,Sales
6,Vikram,Sales
7,Anita,HR
8,Manoj,HR
9,Kavya,Finance


In [5]:
"""-- 3. LEFT JOIN  [Service / joins]
-- Every employee's name with their dept_name,
-- INCLUDING anyone whose department is missing. 15 rows expected."""
query3 = 'Select e.name,d.dept_name from employees e LEFT JOIN departments d on d.dept_id=e.dept_id'
run(query3)

,name,dept_name
0,Arjun,Engineering
1,Priya,Engineering
2,Karan,Engineering
3,Divya,Engineering
4,Rohit,Sales
5,Sneha,Sales
6,Vikram,Sales
7,Anita,HR
8,Manoj,HR
9,Kavya,Finance


In [9]:
"""-- 4. GROUP BY  [Service / aggregation]
-- Count of employees per dept_id."""
query4 = 'Select d.dept_name,e.dept_id,count(e.name) as "COUNT" from employees e left join departments d on d.dept_id=e.dept_id group by e.dept_id'
run(query4)

,dept_name,dept_id,COUNT
0,Engineering,1,5
1,Sales,2,4
2,HR,3,2
3,Finance,4,2
4,Marketing,5,1
5,None,6,1


In [10]:
"""-- 5. GROUP BY + HAVING  [Service / aggregation]
-- dept_id and average salary, only for departments
-- whose average salary is above 65000."""
query5 = 'Select d.dept_name,e.dept_id,AVG(e.salary) as "average_salary" from employees e left join departments d on d.dept_id=e.dept_id group by e.dept_id'
run(query5)

,dept_name,dept_id,average_salary
0,Engineering,1,77200.0
1,Sales,2,66250.0
2,HR,3,48500.0
3,Finance,4,70000.0
4,Marketing,5,59000.0
5,None,6,52000.0


In [11]:
"""-- 6. Scalar subquery  [Service / subquery]
-- Names of employees who earn MORE than the company-wide average salary."""
query6 = 'select name,salary from employees where salary > (select AVG(salary) from employees)'
run(query6)

,name,salary
0,Arjun,95000
1,Priya,72000
2,Karan,68000
3,Divya,81000
4,Sneha,88000
5,Kavya,76000
6,Ramesh,70000


In [14]:
"""-- 7. 3-table join  [AI screen / joins]
-- For each assignment: employee name, project_name, and hours."""
query7="""select e.name,p.project_name,a.hours from assignments a 
JOIN employees e on e.emp_id = a.emp_id
JOIN projects p on p.project_id = a.project_id"""
run(query7)

,name,project_name,hours
0,Arjun,Payment Gateway,120
1,Priya,Payment Gateway,200
2,Karan,Payment Gateway,160
3,Priya,Mobile App,90
4,Divya,Mobile App,210
5,Ramesh,Mobile App,150
6,Deepa,Q4 Campaign,180
7,Rohit,CRM Rollout,140
8,Vikram,CRM Rollout,130
9,Lakshmi,CRM Rollout,100


In [21]:
"""-- 8. Self-join  [AI screen / joins]
-- Each employee's name next to their manager's name.
-- Employees with no manager should still appear (manager name can be NULL)."""
query8 = "select e.Name as 'Manager Name',m.Name from employees e LEFT JOIN employees m on m.manager_id = e.emp_id"
run(query8)

,Manager Name,name
0,Arjun,Divya
1,Arjun,Karan
2,Arjun,Priya
3,Arjun,Ramesh
4,Priya,None
5,Karan,None
6,Divya,None
7,Rohit,None
8,Sneha,Lakshmi
9,Sneha,Rohit


In [22]:
"""-- 9. Window function - RANK  [AI screen / window]
-- Rank employees by salary WITHIN each department
-- (highest = rank 1). Return name, dept_id, salary, rank."""
query9 = """SELECT name,dept_id,salary,rank() over(partition by dept_id order by salary desc) as 'rank' from employees"""
run(query9)

,name,dept_id,salary,rank
0,Arjun,1,95000,1
1,Divya,1,81000,2
2,Priya,1,72000,3
3,Ramesh,1,70000,4
4,Karan,1,68000,5
5,Sneha,2,88000,1
6,Lakshmi,2,62000,2
7,Rohit,2,60000,3
8,Vikram,2,55000,4
9,Anita,3,50000,1
